In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from datasets import load_dataset
import pandas as pd
from PIL import Image

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda', index=0)

In [16]:

class Agent(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = torchvision.models.resnet18()

        self.state = nn.Sequential(
            nn.Linear(1000, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
        )

        self.keyboard = nn.Linear(512, 7)
        self.mouse_buttons = nn.Linear(512, 2)
        self.mouse_pos = nn.Linear(512, 2)

    def forward(self, image):
        x = self.backbone(image)
        x = self.state(x)
        keys = self.keyboard(x)
        mouse_pos = self.mouse_pos(x)
        mouse_buttons = self.mouse_buttons(x)
        
        return mouse_pos, mouse_buttons, keys


In [17]:

model = Agent()
model = model.to(device)

x = torch.randn(1, 3, 224, 224).to(device)

mouse_pos, mouse_buttons, keys = model(x)

print(keys.shape)
print(mouse_pos.shape)
print(mouse_buttons.shape)


torch.Size([1, 7])
torch.Size([1, 2])
torch.Size([1, 2])


In [19]:

# Загружаем датасет
ds = load_dataset(
    "Erik606/roblox-rivals-gameplay-dataset",
    split="train"
)


In [33]:

# Dataset
from io import BytesIO

from torchvision import transforms


class RobloxDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(BytesIO(row["image"])).convert("RGB")

        if self.transform:
            image = self.transform(image)

        mouse_pos = torch.tensor(
            [row["mouse_x"], row["mouse_y"]],
            dtype=torch.float32,
        )

        mouse_buttons = torch.tensor(
            [row["mouse_left"], row["mouse_right"]],
            dtype=torch.float32,
        )

        keys = torch.tensor(
            [
                row["key_w"],
                row["key_a"],
                row["key_s"],
                row["key_d"],
                row["key_space"],
                row["key_shift"],
                row["key_ctrl"],
            ],
            dtype=torch.float32,
        )

        return image, mouse_pos, mouse_buttons, keys


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


dataset = RobloxDataset(ds.to_pandas(), transform)

dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=False)

print(dataset[2]) 
print(dataloader)

(tensor([[[0.1176, 0.1216, 0.1255,  ..., 0.1451, 0.1451, 0.1451],
         [0.1255, 0.1059, 0.1059,  ..., 0.1451, 0.1451, 0.1451],
         [0.1137, 0.0784, 0.1412,  ..., 0.1686, 0.1647, 0.1451],
         ...,
         [0.1176, 0.1490, 0.1412,  ..., 0.3529, 0.2000, 0.1176],
         [0.1176, 0.1216, 0.1216,  ..., 0.1412, 0.1176, 0.1176],
         [0.1176, 0.1176, 0.1216,  ..., 0.1176, 0.1176, 0.1176]],

        [[0.1294, 0.1294, 0.1255,  ..., 0.1451, 0.1451, 0.1451],
         [0.1216, 0.1255, 0.1490,  ..., 0.1451, 0.1451, 0.1451],
         [0.1294, 0.2118, 0.3529,  ..., 0.1686, 0.1647, 0.1451],
         ...,
         [0.1176, 0.1490, 0.1412,  ..., 0.3529, 0.2000, 0.1176],
         [0.1176, 0.1216, 0.1216,  ..., 0.1412, 0.1176, 0.1176],
         [0.1176, 0.1176, 0.1216,  ..., 0.1176, 0.1176, 0.1176]],

        [[0.1098, 0.0980, 0.1020,  ..., 0.1451, 0.1451, 0.1451],
         [0.1176, 0.1294, 0.1686,  ..., 0.1451, 0.1451, 0.1451],
         [0.1373, 0.2745, 0.4667,  ..., 0.1686, 0.1647, 0

In [ ]:

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-2,
)

key_loss = nn.BCEWithLogitsLoss()
mouse_button_loss = nn.BCEWithLogitsLoss()
mouse_pos_loss = nn.SmoothL1Loss()
